In [1]:
#1 install Libraries
#------------------------

!pip install -q langchain langchain-community langchain-text-splitters chromadb pypdf sentence-transformers
!pip install -q langchain langchain-community langchain-chroma chromadb sentence-transformers transformers torch pypdf

In [2]:
#2 Import Libraries
#---------------------

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("Imports Successful")

C:\Users\anamika\AppData\Local\Temp\ipykernel_14576\956173843.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Imports Successful


In [3]:
#3 Load Employee Handbook PDF

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Employee_handbook.pdf")
pages = loader.load()

print("Total Pages Loaded:", len(pages))

Total Pages Loaded: 28


In [4]:
#5 Verify PDF Text
#-------------------

print(pages[0].page_content[:1000])

©2004 National Council of Nonprofit Associations 
May be duplicated, with attribution, by charitable organizations. 
 
 
 
 
 
 
 
 
 
 
SAMPLE EMPLOYEE HANDBOOK


In [5]:
#6 chunking
#---------------

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,chunk_overlap = 200,)
chunks=text_splitter.split_documents(pages)
print("Total Chunks:",len(chunks))

Total Chunks: 93


In [6]:
#7 Verify the First Chunk
#----------------------------

print(chunks[0].page_content)

©2004 National Council of Nonprofit Associations 
May be duplicated, with attribution, by charitable organizations. 
 
 
 
 
 
 
 
 
 
 
SAMPLE EMPLOYEE HANDBOOK


In [7]:
#8 Embedding
#----------------

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Embeddings Loaded")

C:\Users\anamika\AppData\Local\Temp\ipykernel_14576\1959376572.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings Loaded


In [9]:
#9 vector database
#------------------------

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Vector DB Created Successfully")

Vector DB Created Successfully


In [10]:
#10 retriever
#-------------

retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [11]:
#11 Test Retrieval
#----------------

query = "What is the vacation policy?"
docs = retriever.invoke(query)
print(docs[0].page_content)

increments, using the appropriate leave request form..   
 
  Employees are expected to use Vacation benefits in the fiscal year in which Vacation is 
earned.  Employees may carry over unused Vacation from one year to the next only with the 
approval of the Executive Director.  Employees may not accrue more than the maximum leave 
they are allowed.  Once an employee reaches his or her annual ceiling, the employee ceases to 
accrue any additional Vacation benefits.  If an employee later uses enough Vacation benefits to 
fall below the ceiling, the employee starts to accrue leave again from that date forward until he 
or she reaches his or her Vacation ceiling.  Accordingly, employees are encouraged to use all 
Vacation benefits in the fiscal year in which they are earned in order to avoid reaching the 
ceiling limit. 
 
C.  Sick Leave 
  Sick leave benefits are earned on a prorated basis of one day (7 hours) per month for


In [12]:
#12  qn Function
#-------------------

def ask_question(question):
    docs = retriever.invoke(question)

    print("Question:")
    print(question)

    print("\nRetrieved Answer:")
    print(docs[0].page_content)


In [13]:
#13 Function call
#--------------------

ask_question("What are the working hours?")

Question:
What are the working hours?

Retrieved Answer:
7 
 
involving a Sunday or holidays when the rate is two times the regular rate.  Payment of 
overtime will be provided in the pay period following the period in which it is earned. 
  
VIII.  EMPLOYMENT POLICIES AND PRACTICES 
A.  Definition of Terms 
1.  Employer.  The {Organization Name} is the employer of all full‐time, part‐time 
and temporary employees.  An employee is hired, provided compensation and 
applicable benefits, and has his or her work directed and evaluated by 
{ORGANIZATION NAME}. 
 
2.  Full‐Time Employee.  A Full Time Employee regularly works at least 35 hours 
per week 
 
3.  Part‐Time Employee.  A Part Time Employee regularly works less than 35 hours 
per week but no less than 17 ½ hours per week. 
 
4.  Exempt Employee.  An Exempt Employee is an employee who is paid on a salary 
basis and meets the qualifications for exemption from the overtime requirements 
of the Fair Labor Standards Act (“FLSA”).


In [13]:
#-----------------------------------------------------

In [15]:
#14 Reranker
#----------------

from sentence_transformers import CrossEncoder

class Reranker:
    def __init__(self, model_name="BAAI/bge-reranker-base"):
        self.model = CrossEncoder(model_name)

    def rerank(self, query, docs, top_n=3):

        pairs = [[query, doc.page_content] for doc in docs]

        scores = self.model.predict(pairs)

        reranked = sorted(
            zip(docs, scores),
            key=lambda x: x[1],
            reverse=True
        )

        return [doc for doc, score in reranked[:top_n]]

In [ ]:
#15 Retriever
#-----------------

base_retriever = vector_db.as_retriever(search_kwargs={"k": 10})
reranker = Reranker()

print("Reranker Ready")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
#16
#--------

def reranking_retriever(query):

    initial_docs = base_retriever.invoke(query)

    final_docs = reranker.rerank(
        query,
        initial_docs,
        top_n=3
    )

    return final_docs

In [ ]:
docs = reranking_retriever( "What are the working hours?")
print(docs[0].page_content)

6 
 
VII.  HOURS OF WORK, ATTENDANCE AND PUNCTUALITY 
A.  Hours of Work 
The normal work week for {ORGANIZATION NAME} shall consist of five (5), seven (7) 
hour days.  Ordinarily, work hours are from 9:00 a.m. ‐ 5:00 p.m., Monday through Friday, 
including one hour (unpaid) for lunch.  Employees may request the opportunity to vary their 
work schedules (within employer‐defined limits) to better accommodate personal 
responsibilities.  Subject to {ORGANIZATION NAME} work assignments and Executive 
Director approval, the employee’s supervisor shall determine the hours of employment that 
best suits the needs of the work to be done by the individual employee. 
 
B.  Attendance and Punctuality 
Attendance is a key factor in your job performance.  Punctuality and regular attendance 
are expected of all employees.  Excessive absences (whether excused or unexcused), tardiness or 
leaving early is unacceptable.  If you are absent for any reason or plan to arrive late or leave


In [ ]:
#18 prompt Template
#---------------------

from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""
Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer:
""")

print("Prompt Ready")

Prompt Ready


In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
query = "What are the working hours?"

docs = reranking_retriever(query)

context = format_docs(docs)

final_prompt = prompt.format(
    context=context,
    question=query
)

print(final_prompt)


Use the following context to answer the question.

Context:
6 
 
VII.  HOURS OF WORK, ATTENDANCE AND PUNCTUALITY 
A.  Hours of Work 
The normal work week for {ORGANIZATION NAME} shall consist of five (5), seven (7) 
hour days.  Ordinarily, work hours are from 9:00 a.m. ‐ 5:00 p.m., Monday through Friday, 
including one hour (unpaid) for lunch.  Employees may request the opportunity to vary their 
work schedules (within employer‐defined limits) to better accommodate personal 
responsibilities.  Subject to {ORGANIZATION NAME} work assignments and Executive 
Director approval, the employee’s supervisor shall determine the hours of employment that 
best suits the needs of the work to be done by the individual employee. 
 
B.  Attendance and Punctuality 
Attendance is a key factor in your job performance.  Punctuality and regular attendance 
are expected of all employees.  Excessive absences (whether excused or unexcused), tardiness or 
leaving early is unacceptable.  If you are absent for

In [ ]:
!pip install -q transformers accelerate sentencepiece

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

In [ ]:
def ask_rag(question):

    docs = reranking_retriever(question)

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    prompt_text = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

    response = generator(
        prompt_text,
        max_new_tokens=150
    )

    return response[0]["generated_text"]

In [ ]:
print(ask_rag( "What are the working hours?" ))


Token indices sequence length is longer than the specified maximum sequence length for this model (740 > 512). Running this sequence through the model will result in indexing errors
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Use the following context to answer the question.

Context:
6 
 
VII.  HOURS OF WORK, ATTENDANCE AND PUNCTUALITY 
A.  Hours of Work 
The normal work week for {ORGANIZATION NAME} shall consist of five (5), seven (7) 
hour days.  Ordinarily, work hours are from 9:00 a.m. ‐ 5:00 p.m., Monday through Friday, 
including one hour (unpaid) for lunch.  Employees may request the opportunity to vary their 
work schedules (within employer‐defined limits) to better accommodate personal 
responsibilities.  Subject to {ORGANIZATION NAME} work assignments and Executive 
Director approval, the employee’s supervisor shall determine the hours of employment that 
best suits the needs of the work to be done by the individual employee. 
 
B.  Attendance and Punctuality 
Attendance is a key factor in your job performance.  Punctuality and regular attendance 
are expected of all employees.  Excessive absences (whether excused or unexcused), tardiness or 
leaving early is unacceptable.  If you are absent for

In [ ]:
print(ask_rag( "What is the vacation policy?" ))


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Use the following context to answer the question.

Context:
11 
 
  During the first 90 days of employment full‐ and part‐time employees will not earn 
Vacation benefits.  During the remaining nine months of first year employment, a full‐time 
employee will earn two weeks (10 days) of paid Vacation. 
 
  Full‐time employees will continue to earn two weeks (10 days) of Vacation during the 
second year of employment.  In their third year of employment, full‐time employees will earn 
three weeks (15 days) of Vacation.  During the fourth year and thereafter, full‐time employees 
will earn four weeks (20 days) of Vacation per year.  Temporary employees are ineligible for 
Vacation benefits.  
 
  Vacation benefits are prorated accordingly for Part‐Time employees.  Use of Vacation is 
subject to approval by the supervisor and Executive Director and must be requested in hourly 
increments, using the appropriate leave request form..   
 
  Employees are expected to use Vacation benefits in th

**Evaluation Metrics**

In [ ]:
!pip install -q sentence_transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
questions = [
    "What are the working hours?",
    "What is the vacation policy?",
    "What is the sick leave policy?",
    "What are the holidays?",
    "What is the confidentiality policy?"
]

In [ ]:
ground_truths = [
    "The normal work week consists of five seven-hour days.",
    "Employees receive vacation benefits according to company policy.",
    "Employees can use sick leave when they are ill.",
    "Employees are entitled to company holidays.",
    "Employees must maintain confidentiality of company information."
]

In [ ]:
print(ask_rag("What are the working hours?"))

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Use the following context to answer the question.

Context:
6 
 
VII.  HOURS OF WORK, ATTENDANCE AND PUNCTUALITY 
A.  Hours of Work 
The normal work week for {ORGANIZATION NAME} shall consist of five (5), seven (7) 
hour days.  Ordinarily, work hours are from 9:00 a.m. ‐ 5:00 p.m., Monday through Friday, 
including one hour (unpaid) for lunch.  Employees may request the opportunity to vary their 
work schedules (within employer‐defined limits) to better accommodate personal 
responsibilities.  Subject to {ORGANIZATION NAME} work assignments and Executive 
Director approval, the employee’s supervisor shall determine the hours of employment that 
best suits the needs of the work to be done by the individual employee. 
 
B.  Attendance and Punctuality 
Attendance is a key factor in your job performance.  Punctuality and regular attendance 
are expected of all employees.  Excessive absences (whether excused or unexcused), tardiness or 
leaving early is unacceptable.  If you are absent for

In [ ]:
answers = []

for question in questions:
    answer = ask_rag(question)
    answers.append(answer)

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

In [ ]:
scores = []

for i in range(len(answers)):
    similarity = util.cos_sim(
        embedding_model.encode(answers[i]),
        embedding_model.encode(ground_truths[i])
    )

    score = float(similarity) * 100
    scores.append(score)

    print(f"Q{i+1} Score: {score:.2f}%")

Q1 Score: 64.85%
Q2 Score: 66.98%
Q3 Score: 63.06%
Q4 Score: 58.92%
Q5 Score: 67.69%


In [ ]:
average_score = sum(scores) / len(scores)

print(f"Average Accuracy: {average_score:.2f}%")

Average Accuracy: 64.30%
